In [1]:
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C

# Load Function 8 data
X = np.load("function8/initial_inputs.npy")
Y = np.load("function8/initial_outputs.npy")
print("X shape:", X.shape)
print("Y shape:", Y.shape)


print(X)


print(Y)

X shape: (40, 8)
Y shape: (40,)
[[0.60499445 0.29221502 0.90845275 0.35550624 0.20166872 0.57533801
  0.31031095 0.73428138]
 [0.17800696 0.56622265 0.99486184 0.21032501 0.32015266 0.70790879
  0.63538449 0.10713163]
 [0.00907698 0.81162615 0.52052036 0.07568668 0.26511183 0.09165169
  0.59241515 0.36732026]
 [0.50602816 0.65373012 0.36341078 0.17798105 0.0937283  0.19742533
  0.7558269  0.29247234]
 [0.35990926 0.24907568 0.49599717 0.70921498 0.11498719 0.28920692
  0.55729515 0.59388173]
 [0.77881834 0.0034195  0.33798313 0.51952778 0.82090699 0.53724669
  0.5513471  0.66003209]
 [0.90864932 0.0622497  0.23825955 0.76660355 0.13233596 0.99024381
  0.68806782 0.74249594]
 [0.58637144 0.88073573 0.74502075 0.54603485 0.00964888 0.74899176
  0.23090707 0.09791562]
 [0.76113733 0.85467239 0.38212433 0.33735198 0.68970832 0.30985305
  0.63137968 0.04195607]
 [0.9849332  0.69950626 0.9988855  0.18014846 0.58014315 0.23108719
  0.49082694 0.31368272]
 [0.11207131 0.43773566 0.59659878 0.5

In [2]:
best_index = np.argmax(Y)

print("Best index:", best_index)
print("Best current x:", X[best_index])
print("Best current y:", Y[best_index])
print("Best current portal format:", "-".join(f"{v:.6f}" for v in X[best_index]))

kernel = C(1.0) * RBF(length_scale=0.2)

gp = GaussianProcessRegressor(
    kernel=kernel,
    alpha=1e-10,
    normalize_y=True,
    random_state=42
)

gp.fit(X, Y)

rng = np.random.default_rng(42)
candidates = rng.uniform(0, 1, size=(10000, X.shape[1]))

mean, std = gp.predict(candidates, return_std=True)

kappa = 2.5
ucb = mean + kappa * std

best_ucb_index = np.argmax(ucb)
query = candidates[best_ucb_index]

print("Suggested query:", query)
print("Portal format:", "-".join(f"{v:.6f}" for v in query))
print("Predicted mean:", mean[best_ucb_index])
print("Predicted std:", std[best_ucb_index])
print("UCB score:", ucb[best_ucb_index])

Best index: 14
Best current x: [0.05644741 0.06595555 0.02292868 0.03878647 0.40393544 0.80105533
 0.48830701 0.89308498]
Best current y: 9.598482002566342
Best current portal format: 0.056447-0.065956-0.022929-0.038786-0.403935-0.801055-0.488307-0.893085
Suggested query: [0.34445146 0.15640989 0.00666978 0.03381677 0.85385937 0.61485803
 0.04555915 0.10672634]
Portal format: 0.344451-0.156410-0.006670-0.033817-0.853859-0.614858-0.045559-0.106726
Predicted mean: 9.931831924576073
Predicted std: 0.5726826611613081
UCB score: 11.363538577479343


In [3]:
import numpy as np
from scipy.stats import norm
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C

# Load original Function 8 data
X = np.load("function8/initial_inputs.npy")
Y = np.load("function8/initial_outputs.npy")
#For Function 8, I used Expected Improvement because Week 1 improved the best value, suggesting there was a promising region worth testing further. Since Function 8 is high-dimensional, EI was useful because it balanced the predicted score with uncertainty without over-prioritising uncertain areas. 
#The selected query had a predicted mean above the current best, so it was a strong candidate for further improvement.
# Add Week 1 query and output
week1_x = np.array([[0.344451, 0.156410, 0.006670, 0.033817, 0.853859, 0.614858, 0.045559, 0.106726]])
week1_y = np.array([9.7346382479349])

X = np.vstack([X, week1_x])
Y = np.append(Y, week1_y)

print("Updated X shape:", X.shape)
print("Updated Y shape:", Y.shape)

print("Best current x:", X[np.argmax(Y)])
print("Best current y:", np.max(Y))

# Fit GP surrogate model
kernel = C(1.0) * RBF(length_scale=0.2)

gp = GaussianProcessRegressor(
    kernel=kernel,
    alpha=1e-10,
    normalize_y=True,
    random_state=42
)

gp.fit(X, Y)

# Generate candidate points
rng = np.random.default_rng(42)
candidates = rng.uniform(0, 1, size=(10000, X.shape[1]))

# Predict mean and uncertainty
mean, std = gp.predict(candidates, return_std=True)

# Expected Improvement acquisition
y_best = np.max(Y)
std_safe = std + 1e-12

improvement = mean - y_best
z = improvement / std_safe
ei = improvement * norm.cdf(z) + std_safe * norm.pdf(z)

best_ei_index = np.argmax(ei)
query = candidates[best_ei_index]

print("Acquisition used: Expected Improvement")
print("Suggested query:", query)
print("Portal format:", "-".join(f"{v:.6f}" for v in query))
print("Predicted mean:", mean[best_ei_index])
print("Predicted std:", std[best_ei_index])
print("EI score:", ei[best_ei_index])

Updated X shape: (41, 8)
Updated Y shape: (41,)
Best current x: [0.344451 0.15641  0.00667  0.033817 0.853859 0.614858 0.045559 0.106726]
Best current y: 9.7346382479349
Acquisition used: Expected Improvement
Suggested query: [0.14794667 0.19382545 0.06098051 0.00800355 0.71930589 0.45346053
 0.04559219 0.93896982]
Portal format: 0.147947-0.193825-0.060981-0.008004-0.719306-0.453461-0.045592-0.938970
Predicted mean: 10.139947853439969
Predicted std: 0.40901378244751296
EI score: 0.43997853927166897


In [4]:
import numpy as np
from scipy.stats import norm

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel as C


# For Function 8, Week 1 and Week 2 both produced strong outputs, with Week 2 slightly better.
# Because Function 8 is eight-dimensional, I do not want to rely only on local exploitation.
# I use a filtered hybrid EI-UCB strategy with candidates around both strong previous regions,
# plus a larger global candidate pool to maintain exploration in the high-dimensional space.


# -----------------------------
# Load original Function 8 data
# -----------------------------
X = np.load("function8/initial_inputs.npy")
Y = np.load("function8/initial_outputs.npy")


# -----------------------------
# Add Week 1 and Week 2 results
# -----------------------------
week1_x = np.array([[0.344451, 0.156410, 0.006670, 0.033817, 0.853859, 0.614858, 0.045559, 0.106726]])
week1_y = np.array([9.7346382479349])

week2_x = np.array([[0.147947, 0.193825, 0.060981, 0.008004, 0.719306, 0.453461, 0.045592, 0.938970]])
week2_y = np.array([9.894432442301])

X = np.vstack([X, week1_x, week2_x])
Y = np.append(Y, [week1_y[0], week2_y[0]])


print("Updated X shape:", X.shape)
print("Updated Y shape:", Y.shape)

best_index = np.argmax(Y)
best_x = X[best_index]
best_y = Y[best_index]

print("Best current x:", best_x)
print("Best current y:", best_y)


# -----------------------------
# Fit GP surrogate model
# -----------------------------
kernel = (
    C(1.0, (1e-3, 1e3))
    * Matern(length_scale=np.ones(X.shape[1]) * 0.2, length_scale_bounds=(1e-2, 1.0), nu=2.5)
    + WhiteKernel(noise_level=1e-8, noise_level_bounds=(1e-10, 1e-3))
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

gp.fit(X, Y)

print("Fitted kernel:", gp.kernel_)


# -----------------------------
# Generate candidate points
# -----------------------------
rng = np.random.default_rng(42)
dim = X.shape[1]

# Function 8 is high-dimensional, so use a larger global pool.
global_candidates = rng.uniform(0, 1, size=(50000, dim))

# Local candidates around the current best Week 2 point.
week2_candidates = rng.normal(loc=week2_x[0], scale=0.10, size=(30000, dim))
week2_candidates = np.clip(week2_candidates, 0, 1)

# Also include candidates around Week 1 because it was also strong.
week1_candidates = rng.normal(loc=week1_x[0], scale=0.10, size=(20000, dim))
week1_candidates = np.clip(week1_candidates, 0, 1)

candidates = np.vstack([global_candidates, week2_candidates, week1_candidates])


# -----------------------------
# Predict mean and uncertainty
# -----------------------------
mean, std = gp.predict(candidates, return_std=True)

y_best = np.max(Y)
std_safe = std + 1e-12


# -----------------------------
# Expected Improvement
# -----------------------------
improvement = mean - y_best
z = improvement / std_safe
ei = improvement * norm.cdf(z) + std_safe * norm.pdf(z)


# -----------------------------
# Upper Confidence Bound
# -----------------------------
# Moderate kappa because Function 8 already has strong values,
# but the 8D space still needs exploration.
kappa = 1.6
ucb = mean + kappa * std


# -----------------------------
# Filtered hybrid EI-UCB
# -----------------------------
# Keep candidates that are predicted to be competitive.
# The threshold is looser because in 8D the model can be uncertain.
mean_filter = mean >= (y_best - 0.25)

if np.sum(mean_filter) == 0:
    mean_filter = mean >= np.percentile(mean, 90)

filtered_candidates = candidates[mean_filter]
filtered_mean = mean[mean_filter]
filtered_std = std[mean_filter]
filtered_ei = ei[mean_filter]
filtered_ucb = ucb[mean_filter]

ei_norm = (filtered_ei - np.min(filtered_ei)) / (np.max(filtered_ei) - np.min(filtered_ei) + 1e-12)
ucb_norm = (filtered_ucb - np.min(filtered_ucb)) / (np.max(filtered_ucb) - np.min(filtered_ucb) + 1e-12)

# EI is weighted more heavily because there is already a strong best value,
# but UCB remains meaningful because the space is 8D.
hybrid_score = 0.65 * ei_norm + 0.35 * ucb_norm

best_hybrid_index = np.argmax(hybrid_score)
query = filtered_candidates[best_hybrid_index]


# -----------------------------
# Output results
# -----------------------------
print("Acquisition used: Filtered Hybrid EI + UCB")
print("Number of candidates passing filter:", np.sum(mean_filter))
print("Suggested query:", query)

print("Portal format with hyphens:")
print("-".join(f"{v:.6f}" for v in query))

print("Portal format with x labels:")
print(",".join(f"x{i+1}:{v:.6f}" for i, v in enumerate(query)))

print("Predicted mean:", filtered_mean[best_hybrid_index])
print("Predicted std:", filtered_std[best_hybrid_index])
print("EI score:", filtered_ei[best_hybrid_index])
print("UCB score:", filtered_ucb[best_hybrid_index])
print("Hybrid score:", hybrid_score[best_hybrid_index])

Updated X shape: (42, 8)
Updated Y shape: (42,)
Best current x: [0.147947 0.193825 0.060981 0.008004 0.719306 0.453461 0.045592 0.93897 ]
Best current y: 9.894432442301


/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 1.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 3 of parameter k1__k2__length_scale is close to the specified upper bound 1.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 4 of parameter k1__k2__length_scale is close to the specified upper bound 1.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/co

Fitted kernel: 0.774**2 * Matern(length_scale=[0.794, 1, 0.595, 1, 1, 1, 0.884, 1], nu=2.5) + WhiteKernel(noise_level=6.47e-10)
Acquisition used: Filtered Hybrid EI + UCB
Number of candidates passing filter: 43685
Suggested query: [0.12911283 0.20731767 0.16505186 0.2441378  0.60364633 0.53575882
 0.16863045 0.6815165 ]
Portal format with hyphens:
0.129113-0.207318-0.165052-0.244138-0.603646-0.535759-0.168630-0.681517
Portal format with x labels:
x1:0.129113,x2:0.207318,x3:0.165052,x4:0.244138,x5:0.603646,x6:0.535759,x7:0.168630,x8:0.681517
Predicted mean: 10.143484541227666
Predicted std: 0.24916622860433665
EI score: 0.26982961411630874
UCB score: 10.542150506994606
Hybrid score: 0.9775672220064827


In [5]:
week3_x = np.array([[
    0.129113,
    0.207318,
    0.165052,
    0.244138,
    0.603646,
    0.535759,
    0.168630,
    0.681517
]])

week3_y = np.array([9.9592828135141])

X = np.vstack([X, week1_x, week2_x, week3_x])
Y = np.append(Y, [
    week1_y[0],
    week2_y[0],
    week3_y[0]
])

best_index = np.argmax(Y)
best_x = X[best_index]
best_y = Y[best_index]

print("Shape:", X.shape, Y.shape)
print("Current best x:", best_x)
print("Current best y:", best_y)
print("Was Week 3 best?", best_index == len(Y) - 1)

Shape: (45, 8) (45,)
Current best x: [0.129113 0.207318 0.165052 0.244138 0.603646 0.535759 0.16863  0.681517]
Current best y: 9.9592828135141
Was Week 3 best? True


In [6]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel as C,
    Matern,
    WhiteKernel
)

kernel = (
    C(1.0, (1e-3, 1e3))
    * Matern(
        length_scale=np.full(8, 0.2),
        length_scale_bounds=(1e-2, 2.0),
        nu=2.5
    )
    + WhiteKernel(
        noise_level=1e-8,
        noise_level_bounds=(1e-10, 1e-3)
    )
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=40,
    random_state=42
)

gp.fit(X, Y)

print("Fitted kernel:", gp.kernel_)

/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/_gpr.py:663: ConvergenceWarning: lbfgs failed to converge after 30 iteration(s) (status=2):
ABNORMAL: 

You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  _check_optimize_result("lbfgs", opt_res)
/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/_gpr.py:663: ConvergenceWarning: lbfgs failed to converge after 26 iteration(s) (status=2):
ABNORMAL: 

You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  _check_optimize_result("lbfgs", opt_res)
/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/_gpr.py:663: ConvergenceWarning: lbfgs failed to converge after 15 iteration(s) (status=2):
ABNORMAL: 

You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/

Fitted kernel: 0.973**2 * Matern(length_scale=[1.22, 2, 0.947, 2, 2, 2, 1.32, 2], nu=2.5) + WhiteKernel(noise_level=1e-10)


/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 3 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 4 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/co

In [7]:
trust_radius = np.full(8, 0.10)

lower_bounds = np.maximum(0, best_x - trust_radius)
upper_bounds = np.minimum(1, best_x + trust_radius)

bounds = list(zip(lower_bounds, upper_bounds))

print("Lower bounds:", lower_bounds)
print("Upper bounds:", upper_bounds)

Lower bounds: [0.029113 0.107318 0.065052 0.144138 0.503646 0.435759 0.06863  0.581517]
Upper bounds: [0.229113 0.307318 0.265052 0.344138 0.703646 0.635759 0.26863  0.781517]


In [8]:
kappa = 0.6

def negative_ucb(point):
    point = np.asarray(point).reshape(1, -1)

    mean, std = gp.predict(point, return_std=True)
    ucb = mean[0] + kappa * std[0]

    return -ucb

In [9]:
from scipy.optimize import minimize

rng = np.random.default_rng(42)

random_starts = rng.uniform(
    lower_bounds,
    upper_bounds,
    size=(250, 8)
)

starting_points = [
    best_x,
    np.clip(week3_x[0], lower_bounds, upper_bounds)
]

starting_points.extend(random_starts)

results = []

for start in starting_points:
    result = minimize(
        negative_ucb,
        x0=start,
        method="L-BFGS-B",
        bounds=bounds
    )

    if result.success:
        results.append(result)

if not results:
    raise RuntimeError("No successful optimisation runs")

best_result = min(results, key=lambda result: result.fun)
query = best_result.x

In [10]:
query_mean, query_std = gp.predict(
    query.reshape(1, -1),
    return_std=True
)

query_ucb = query_mean[0] + kappa * query_std[0]

distances = np.linalg.norm(X - query, axis=1)
nearest_index = np.argmin(distances)
nearest_distance = distances[nearest_index]

print("Method: Focused trust-region GP-UCB")
print("Suggested Week 4 query:", query)

print(
    "Portal format:",
    "-".join(f"{value:.6f}" for value in query)
)

print("Current best observed output:", best_y)
print("Predicted mean:", query_mean[0])
print("Predicted std:", query_std[0])
print("UCB:", query_ucb)
print("Distance to nearest observation:", nearest_distance)
print("Nearest observed point:", X[nearest_index])
print("Nearest observed output:", Y[nearest_index])

Method: Focused trust-region GP-UCB
Suggested Week 4 query: [0.07849979 0.107318   0.21261904 0.144138   0.703646   0.48532528
 0.13450732 0.581517  ]
Portal format: 0.078500-0.107318-0.212619-0.144138-0.703646-0.485325-0.134507-0.581517
Current best observed output: 9.9592828135141
Predicted mean: 10.033086026540602
Predicted std: 0.11559743798409049
UCB: 10.102444489331056
Distance to nearest observation: 0.22030033628762388
Nearest observed point: [0.129113 0.207318 0.165052 0.244138 0.603646 0.535759 0.16863  0.681517]
Nearest observed output: 9.9592828135141
